# Thinking mode: winning on gradient boosting's home turf

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innixma/priorlabs_tabpfn_demo/blob/main/notebooks/01_thinking_mode.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-181717?logo=github)](https://github.com/Innixma/priorlabs_tabpfn_demo)

*Amazon employee access* is a famous holdout of the pre-foundation-model era: ~33k rows
whose nine features are all high-cardinality categorical codes (resource ids, manager
ids, role codes). Wide categorical spaces like this are gradient boosting's home turf --
the corner of tabular learning where trees have historically been hardest to beat.

This notebook runs that matchup live on one fixed split: standalone XGBoost and
CatBoost against [TabPFN-3](https://priorlabs.ai/technical-reports/tabpfn-3) with
[thinking mode](https://docs.priorlabs.ai/capabilities/thinking-mode) -- one fit per
cell, then all three side by side.

> **Runtime**: ~10 minutes end to end; the CatBoost fit takes a couple of minutes on CPU,
> and the thinking-mode fit runs on the TabPFN API (about 5 minutes at high effort), so
> no GPU is needed.

## Setup

The install cell below is collapsed -- expand it to see the details. It is skipped
outside Colab so it never overwrites a locally managed environment.

In [ ]:
# @title Install dependencies { display-mode: "form" }
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    !command -v uv >/dev/null || pip install -q uv
    !uv pip install -q --python {__import__('sys').executable} openml xgboost catboost scikit-learn tabpfn-client

    # The install may replace Colab's preinstalled numpy; the copy already loaded in this
    # kernel then no longer matches the files on disk and imports break. When that happens,
    # restart the runtime once (continue from the next cell after it reconnects).
    import importlib.metadata
    import numpy
    if importlib.metadata.version("numpy") != numpy.__version__:
        print("numpy changed -- restarting the Colab runtime; re-run FROM THE NEXT CELL when it reconnects.")
        import os
        os.kill(os.getpid(), 9)

## The dataset

One fixed train/test split of the *Amazon_employee_access* task, shared by every model
below. The cell loads a copy bundled with this repository by default; flip `DATA_SOURCE`
to `openml` to fetch the original OpenML task instead (same data, same split).

In [ ]:
# @title Load the dataset { display-mode: "form" }
DATA_SOURCE = "github"  # @param ["github", "openml"]

import numpy as np
import pandas as pd

if DATA_SOURCE == "github":
    url = "https://raw.githubusercontent.com/Innixma/priorlabs_tabpfn_demo/main/data/amazon_employee_access.csv.gz"
    df = pd.read_csv(url)
    # Every feature is a categorical id code; restore the categorical-of-strings dtypes
    # the OpenML loader produces so both sources feed the models identically.
    X = df.drop(columns=["target", "split"]).astype(str).astype("category")
    y = df["target"]
    train_idx = np.where(df["split"] == "train")[0]
    test_idx = np.where(df["split"] == "test")[0]
else:
    import openml

    task = openml.tasks.get_task(363613)  # Amazon_employee_access
    X, y = task.get_X_and_y(dataset_format="dataframe")
    train_idx, test_idx = task.get_train_test_split_indices(repeat=0, fold=0)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
# Binary 0/1 targets for the sklearn-style APIs below.
y_train_bin = (y_train == y_train.cat.categories[1]).astype(int) if hasattr(y_train, "cat") else y_train
y_test_bin = (y_test == y_test.cat.categories[1]).astype(int) if hasattr(y_test, "cat") else y_test

print(f"train: {X_train.shape}, test: {X_test.shape}")
X_train.head(3)

## TabPFN access token

TabPFN-3 runs through the TabPFN API, unlocked by a free Prior Labs access token:

1. Sign up / log in at [ux.priorlabs.ai](https://ux.priorlabs.ai)
2. Accept the license at [ux.priorlabs.ai/account/licenses](https://ux.priorlabs.ai/account/licenses)
3. Copy your access token from [ux.priorlabs.ai/account](https://ux.priorlabs.ai/account)

Tip: save it as a Colab secret named `TABPFN_TOKEN` to skip the prompt next time.

In [ ]:
import getpass
import os

tabpfn_token = os.environ.get("TABPFN_TOKEN")
if not tabpfn_token:
    try:
        from google.colab import userdata
        tabpfn_token = userdata.get("TABPFN_TOKEN")
    except Exception:
        pass
while not tabpfn_token:
    tabpfn_token = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ").strip()
os.environ["TABPFN_TOKEN"] = tabpfn_token
print("token set")

## The matchup -- three fits, identical data

Everything out of the box, one cell per model, each reporting its own test AUC and fit
time:

- **XGBoost** and **CatBoost**, both with native categorical handling -- CatBoost in
  particular is the reference method for data like this.
- **TabPFN-3 with thinking mode** (`thinking_mode=True`): additional inference-time
  computation on top of TabPFN-3, steered toward the metric you declare -- about five
  minutes here, on the TabPFN API.

In [ ]:
import time

from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Features are id codes: mark them categorical on the full frame so train/test agree.
X_cat = X.astype("category")

results = globals().get("results", {})
xgb = XGBClassifier(tree_method="hist", enable_categorical=True, eval_metric="auc")
t0 = time.time()
xgb.fit(X_cat.iloc[train_idx], y_train_bin)
fit_s = time.time() - t0
auc = roc_auc_score(y_test_bin, xgb.predict_proba(X_cat.iloc[test_idx])[:, 1])
results["XGBoost"] = (auc, fit_s)
print(f"XGBoost: AUC = {auc:.4f}   (fit {fit_s:.0f}s)")

In [ ]:
import time

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# CatBoost takes categorical features as strings.
X_str = X.astype(str)

results = globals().get("results", {})
cat = CatBoostClassifier(verbose=0, cat_features=list(X_str.columns))
t0 = time.time()
cat.fit(X_str.iloc[train_idx], y_train_bin)
fit_s = time.time() - t0
auc = roc_auc_score(y_test_bin, cat.predict_proba(X_str.iloc[test_idx])[:, 1])
results["CatBoost"] = (auc, fit_s)
print(f"CatBoost: AUC = {auc:.4f}   (fit {fit_s:.0f}s)")

In [ ]:
import os
import time

import tabpfn_client
from sklearn.metrics import roc_auc_score
from tabpfn_client import TabPFNClassifier

tabpfn_client.set_access_token(os.environ["TABPFN_TOKEN"])

results = globals().get("results", {})
clf = TabPFNClassifier(
    thinking_mode=True,
    thinking_effort="high",
    thinking_metric="roc_auc",
)
t0 = time.time()
clf.fit(X_train, y_train_bin)
fit_s = time.time() - t0
auc = roc_auc_score(y_test_bin, clf.predict_proba(X_test)[:, 1])
results["TabPFN-3 (thinking)"] = (auc, fit_s)
print(f"TabPFN-3 (thinking): AUC = {auc:.4f}   (fit {fit_s:.0f}s)")

## Results

All three, side by side.

In [ ]:
print(f"{'model':<22} {'test AUC':>9} {'fit time':>10}")
for name, (auc, fit_s) in sorted(results.items(), key=lambda kv: kv[1][0], reverse=True):
    print(f"{name:<22} {auc:>9.4f} {fit_s:>9.0f}s")

## What just happened

On wide, high-cardinality categorical data, the strongest gradient boosting has
historically been the wall in-context learning couldn't climb -- and CatBoost duly tops
the classical field here. Thinking mode climbs the wall: TabPFN-3 with one switch moves
past even CatBoost, on the kind of data that was long considered out of reach for
tabular foundation models.

The takeaways for practice:

- Declare the metric you care about (`thinking_metric="roc_auc"` here) -- thinking mode
  steers its extra inference-time computation toward it.
- Thinking mode trades inference-time compute for prediction quality; reach for it when a
  dataset sits in a known foundation-model weak spot or when the last points of a metric
  are valuable. Details and options (`thinking_effort`, `thinking_timeout_s`) are in the
  [docs](https://docs.priorlabs.ai/capabilities/thinking-mode); the model itself is
  described in the [TabPFN-3 technical report](https://priorlabs.ai/technical-reports/tabpfn-3).
- Thinking mode composes with TabPFN-3's native text-feature support, so a single call
  can handle mixed numerical, categorical, and text columns under the same
  inference-time-compute regime.
- It achieves this relying only on TabPFN -- no LLMs, no real data, no internet search,
  and no other model involved.
- It runs through the TabPFN API (`tabpfn-client`), so it works from any laptop -- no GPU
  required.